In [4]:
from sqlalchemy import create_engine, Column, Integer, String, Float, DateTime
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker
from pathlib import Path
import pandas as pd

cwd = Path.cwd()
db_path = cwd / 'data/housing_market.db'
engine = create_engine(f'sqlite:///{db_path}?timeout=30', connect_args={'timeout': 30})
Session = sessionmaker(bind=engine)
Base = declarative_base()

class MedianPrice(Base):
    __tablename__ = 'median_prices'
    id = Column(Integer, primary_key=True)
    RegionID = Column(Integer)
    SizeRank = Column(Integer)
    RegionName = Column(String)
    RegionType = Column(String)
    StateName = Column(String)
    date = Column(DateTime)
    median_price = Column(Float)

class ZHVI(Base):
    __tablename__ = 'zhvi'
    id = Column(Integer, primary_key=True)
    RegionID = Column(Integer)
    SizeRank = Column(Integer)
    RegionName = Column(String)
    RegionType = Column(String)
    StateName = Column(String)
    date = Column(DateTime)
    zhvi = Column(Float)

class RedfinIndex(Base):
    __tablename__ = 'redfin_index'
    id = Column(Integer, primary_key=True)
    region_name = Column(String)
    date = Column(DateTime)
    redfin_hpi_yoy = Column(Float)
    redfin_hpi_mom = Column(Float)

class RedfinMarketTracker(Base):
    __tablename__ = 'redfin_market_tracker'
    id = Column(Integer, primary_key=True)
    period_begin = Column(DateTime)
    period_end = Column(DateTime)
    region = Column(String)
    median_sale_price = Column(Float)
    homes_sold = Column(Integer)
    new_listings = Column(Integer)
    inventory = Column(Integer)
    property_type = Column(String)
    last_updated = Column(DateTime)

# Create tables
Base.metadata.create_all(engine)
print("Database tables created successfully!")

Database tables created successfully!


/var/folders/cr/xvl6gw5n6vx9yv0syd95dp5c0000gn/T/ipykernel_66932/1574124750.py:11: MovedIn20Warning: The ``declarative_base()`` function is now available as sqlalchemy.orm.declarative_base(). (deprecated since: 2.0) (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  Base = declarative_base()


In [5]:
# Load CSV data into database
cwd = Path.cwd()
price_path = cwd / 'data' / 'Metro_median_sale_price_now_uc_sfrcondo_month.csv'
zhvi_path = cwd / 'data' / 'Metro_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month (2).csv'
redfin_index_path = cwd / 'data' / 'Redfin_Home_Price_Index.csv'
redfin_tracker_path = cwd / 'data' / 'redfin_metro_market_tracker_small.tsv'

session = Session()

# Clear existing data
session.query(MedianPrice).delete()
session.query(ZHVI).delete()
session.query(RedfinIndex).delete()
session.query(RedfinMarketTracker).delete()
session.commit()

# Load Median Prices
price_df = pd.read_csv(price_path)
metadf_cols = ['RegionID', 'SizeRank', 'RegionName', 'RegionType', 'StateName']
price_long = price_df.melt(id_vars=metadf_cols, var_name='date', value_name='median_price')
price_long['date'] = pd.to_datetime(price_long['date'])

for _, row in price_long.iterrows():
    record = MedianPrice(
        RegionID=row['RegionID'],
        SizeRank=row['SizeRank'],
        RegionName=row['RegionName'],
        RegionType=row['RegionType'],
        StateName=row['StateName'],
        date=row['date'],
        median_price=row['median_price']
    )
    session.add(record)
session.commit()
print(f"Loaded {session.query(MedianPrice).count()} median price records")

# Load ZHVI
zhvi_df = pd.read_csv(zhvi_path)
zhvi_long = zhvi_df.melt(id_vars=metadf_cols, var_name='date', value_name='zhvi')
zhvi_long['date'] = pd.to_datetime(zhvi_long['date'])

for _, row in zhvi_long.iterrows():
    record = ZHVI(
        RegionID=row['RegionID'],
        SizeRank=row['SizeRank'],
        RegionName=row['RegionName'],
        RegionType=row['RegionType'],
        StateName=row['StateName'],
        date=row['date'],
        zhvi=row['zhvi']
    )
    session.add(record)
session.commit()
print(f"Loaded {session.query(ZHVI).count()} ZHVI records")

# Load Redfin Index
redfin_index = pd.read_csv(redfin_index_path, encoding='utf-16', sep='\t')
redfin_index['Date'] = pd.to_datetime(redfin_index['Month, Year of Date'], format='%B %Y')
redfin_index['redfin_hpi_yoy_numeric'] = redfin_index['Redfin HPI YoY'].str.rstrip('%').astype(float)
redfin_index['redfin_hpi_mom_numeric'] = redfin_index['Redfin HPI MoM'].str.rstrip('%').astype(float)

for _, row in redfin_index.iterrows():
    record = RedfinIndex(
        region_name=row['Region Name'],
        date=row['Date'],
        redfin_hpi_yoy=row['redfin_hpi_yoy_numeric'],
        redfin_hpi_mom=row['redfin_hpi_mom_numeric']
    )
    session.add(record)
session.commit()
print(f"Loaded {session.query(RedfinIndex).count()} Redfin Index records")

# load Redfin Market Tracker
tracker_df = pd.read_csv(redfin_tracker_path, sep='\t')
tracker_df["PERIOD_BEGIN"] = pd.to_datetime(tracker_df["PERIOD_BEGIN"])
tracker_df["PERIOD_END"] = pd.to_datetime(tracker_df["PERIOD_END"])
tracker_df["LAST_UPDATED"] = pd.to_datetime(tracker_df["LAST_UPDATED"], errors="coerce")
tracker_df["REGION"] = tracker_df["REGION"].str.replace(" metro area", "", regex=False).str.strip()

for _, row in tracker_df.iterrows():
    record = RedfinMarketTracker(
        period_begin=row["PERIOD_BEGIN"],
        period_end=row["PERIOD_END"],
        region=row["REGION"],
        median_sale_price=row.get("MEDIAN_SALE_PRICE"),
        homes_sold=row.get("HOMES_SOLD"),
        new_listings=row.get("NEW_LISTINGS"),
        inventory=row.get("INVENTORY"),
        property_type=row.get("PROPERTY_TYPE"),
        last_updated=row["LAST_UPDATED"]
    )
    session.add(record)
session.commit()
print(f"Loaded {session.query(RedfinMarketTracker).count()} Redfin Market Tracker records")

# Close session to prevent locking
session.close()
print("\nDatabase population complete! Session closed.")


Loaded 83709 median price records
Loaded 277450 ZHVI records
Loaded 8300 Redfin Index records
Loaded 500 Redfin Market Tracker records

Database population complete! Session closed.


In [6]:
# Create a fresh session for querying
session = Session()

# get median prices from database
price_records = session.query(MedianPrice).all()
price_df = pd.DataFrame([{
    'RegionID': r.RegionID,
    'SizeRank': r.SizeRank,
    'RegionName': r.RegionName,
    'RegionType': r.RegionType,
    'StateName': r.StateName,
    'date': r.date,
    'median_price': r.median_price
} for r in price_records])

# get ZHVI from database
zhvi_records = session.query(ZHVI).all()
zhvi_df = pd.DataFrame([{
    'RegionID': r.RegionID,
    'SizeRank': r.SizeRank,
    'RegionName': r.RegionName,
    'RegionType': r.RegionType,
    'StateName': r.StateName,
    'date': r.date,
    'zhvi': r.zhvi
} for r in zhvi_records])